# 民声智理 · Google Colab GPU 算力节点部署套件

本 Notebook 用于在 Google Colab（推荐 **L4 24GB** / **A100 40GB** / **T4 16GB** GPU）或任意 Linux GPU 环境上一键拉起与本地/线上环境完全对齐的双引擎服务：

1. **System-2 认知大模型服务**（端口 `8132`，默认使用 `ninfer-serve` 运行 **Ternary Bonsai 2.5 27B PQ2_0** + MTP 投机解码，`model-id: bonsai-2-27b`，100% OpenAI 兼容 `/v1/chat/completions` 与 `/v1/models`）
2. **BAAI/bge-m3 向量嵌入服务**（端口 `8133`，100% OpenAI 兼容 `/v1/embeddings`、`/v1/models` 及单域名别名 `/embed/v1/*`，1024 维）
3. **公网/内网穿透隧道**（支持 `named` 单域名 Cloudflare Named Tunnel：`https://civic-colab.firetable.tech/v1` + `/embed/v1`，以及 `cloudflared` 临时隧道或 `tailscale` 加密内网）
4. **强制 Bearer Token 鉴权**（支持环境变量 / Colab Secrets 自动读取 `CIVIC_COLAB_API_KEY`，未手动指定时自动生成高强度随机密钥）

> 请先在 Colab 菜单栏确认：**代码执行程序 (Runtime) → 更改运行时类型 (Change runtime type) → GPU**。

## Step 1：检查 GPU 型号与可用显存 (`nvidia-smi`)

默认 `ninfer` 后端将自动下载 `Ternary-Bonsai-2.5-27B-PQ2_0-ninfer-v3.ninfer`（7.9 GB）并按显存自动调优：
- **A100 80GB+**：`bonsai-2-27b`（`ctx=65536`, `kv=131072 rk8v4`, `concurrency=4`）
- **A100 40GB**：`bonsai-2-27b`（`ctx=65536`, `kv=65536 rk8v4`, `concurrency=4`）
- **L4 24GB**：`bonsai-2-27b`（`ctx=32768`, `kv=65536 rk4v4`, `concurrency=4`）
- **T4 16GB**：`bonsai-2-27b`（`ctx=32768`, `kv=32768 rk4v4`, `concurrency=2`）

若选择备选 `vllm` / `llama-cpp` 后端，则回退按显存自动匹配 Qwen2.5 系列。

In [ ]:
!nvidia-smi

import shutil, subprocess
if shutil.which("nvidia-smi"):
    info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip()
    print("\n检测到 GPU:", info)
else:
    print("\n⚠️ 未检测到 NVIDIA GPU，请在 Colab 切换至 GPU 运行时。")

## Step 2：安装依赖与准备 `serve_colab.py`

- 默认推荐 **`ninfer`**：仅需安装轻量 Python 依赖（`sentence-transformers`、`fastapi`、`uvicorn`、`huggingface_hub`、`pydantic`），无需漫长编译/安装 `vllm`，也不会和 Colab 预装 PyTorch 冲突；运行时自动下载预编译 CUDA `ninfer-serve` 与 Bonsai 2.5 权重。
- 备选后端：`vllm` 或 `llama-cpp`。
- 按需安装隧道工具：`cloudflared` / `named`（均使用 `cloudflared` 二进制，支持 Quick Tunnel 或单域名固定 Named Tunnel）或 `tailscale`。
- 若当前目录尚未放置 `serve_colab.py`，可在下方提示时直接上传仓库中的 `scripts/colab/serve_colab.py`。

In [ ]:
#@title 安装轻量化运行依赖、Embedding 服务与隧道组件
INSTALL_BACKEND = "ninfer" #@param ["ninfer", "vllm", "llama-cpp"]
INSTALL_TUNNEL = "cloudflared" #@param ["cloudflared", "named", "tailscale", "none"]

import pathlib, shutil, subprocess, sys

base_pkgs = ["sentence-transformers", "fastapi", "uvicorn", "huggingface_hub", "pydantic"]
if INSTALL_BACKEND == "vllm":
    base_pkgs.append("vllm")
elif INSTALL_BACKEND == "llama-cpp":
    base_pkgs.append("llama-cpp-python")

print("正在安装 Python 依赖:", ", ".join(base_pkgs))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *base_pkgs])

if INSTALL_TUNNEL in ("cloudflared", "named") and not shutil.which("cloudflared"):
    print("正在安装 cloudflared...")
    subprocess.check_call([
        "bash", "-lc",
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb && dpkg -i /tmp/cloudflared.deb && rm -f /tmp/cloudflared.deb"
    ])
elif INSTALL_TUNNEL == "tailscale" and not shutil.which("tailscale"):
    print("正在安装 tailscale...")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://tailscale.com/install.sh | sh"])

# 检查 serve_colab.py 是否已存在于工作区
candidates = [
    pathlib.Path("serve_colab.py"),
    pathlib.Path("scripts/colab/serve_colab.py"),
    pathlib.Path("/content/serve_colab.py"),
]
found_script = next((p for p in candidates if p.is_file()), None)
if found_script:
    print(f"✅ 已定位启动脚本: {found_script.resolve()}")
else:
    print("未检测到 serve_colab.py，若在 Google Colab 中请通过文件上传窗口上传 scripts/colab/serve_colab.py：")
    try:
        from google.colab import files
        uploaded = files.upload()
        if "serve_colab.py" in uploaded:
            print("✅ 已上传 serve_colab.py")
    except ImportError:
        print("当前非 Colab 环境，请确保 scripts/colab/serve_colab.py 位于工作目录中。")

## Step 3：配置参数（Token、模型选择、隧道模式）

- **`LLM_BACKEND`**：默认 **`ninfer`**，自动拉取 `ninfer-serve` 与 `WaveCut/Ternary-Bonsai-2.5-27B-PQ2_0-NInfer-v3` 并暴露标准模型名 **`bonsai-2-27b`**。
- **`API_KEY`**：留空将优先读取环境变量 `CIVIC_COLAB_API_KEY` / `COLAB_API_KEY` 或 Colab Secrets，否则通过 `secrets.token_urlsafe(24)` 自动生成高强度随机 Bearer Token（切勿在公开 Notebook 中硬编码真实密钥）。
- **`LLM_MODEL`**：留空则自动使用对齐的 Bonsai 2.5 27B（`ninfer` 模式）或按 GPU 显存推荐的 Qwen2.5 模型（`vllm`/`llama-cpp` 模式）。
- **`TUNNEL_MODE`**：支持 **`named`**（单域名固定 Cloudflare Named Tunnel：`https://civic-colab.firetable.tech/v1` + `/embed/v1`，需在 Colab Secrets 中配置 `CIVIC_COLAB_TUNNEL_TOKEN`）、**`cloudflared`**（临时 HTTPS Quick Tunnel）或 **`tailscale`**。

In [ ]:
#@title 配置 Colab 算力节点参数
LLM_BACKEND = "ninfer" #@param ["ninfer", "vllm", "llama-cpp"]
LLM_MODEL = "" #@param {type:"string"}
EMBED_MODEL = "BAAI/bge-m3" #@param {type:"string"}
TUNNEL_MODE = "cloudflared" #@param ["named", "cloudflared", "tailscale", "none"]
PUBLIC_HOST = "civic-colab.firetable.tech" #@param {type:"string"}
API_KEY = "" #@param {type:"string"}
TUNNEL_TOKEN = "" #@param {type:"string"}
LLM_PORT = 8132 #@param {type:"integer"}
EMBED_PORT = 8133 #@param {type:"integer"}
EMBED_DEVICE = "auto" #@param ["auto", "cuda", "cpu"]

import os, secrets

def _read_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

resolved_key = (
    API_KEY.strip()
    or os.environ.get("CIVIC_COLAB_API_KEY", "").strip()
    or os.environ.get("COLAB_API_KEY", "").strip()
    or _read_colab_secret("CIVIC_COLAB_API_KEY")
    or secrets.token_urlsafe(24)
)
os.environ["COLAB_API_KEY"] = resolved_key
os.environ["CIVIC_COLAB_API_KEY"] = resolved_key
os.environ["CIVIC_COLAB_HOSTNAME"] = PUBLIC_HOST.strip() or "civic-colab.firetable.tech"

resolved_tunnel_token = (
    TUNNEL_TOKEN.strip()
    or os.environ.get("CIVIC_COLAB_TUNNEL_TOKEN", "").strip()
    or os.environ.get("COLAB_TUNNEL_TOKEN", "").strip()
    or _read_colab_secret("CIVIC_COLAB_TUNNEL_TOKEN")
)
if resolved_tunnel_token:
    os.environ["CIVIC_COLAB_TUNNEL_TOKEN"] = resolved_tunnel_token
if TUNNEL_MODE == "named" and not resolved_tunnel_token:
    raise RuntimeError("TUNNEL_MODE='named' 需要配置 CIVIC_COLAB_TUNNEL_TOKEN（推荐放入 Colab Secrets）。")

masked = f"****{resolved_key[-4:]}" if len(resolved_key) >= 4 else "****"
print(f"✅ 参数已就绪:")
print(f"  • LLM_BACKEND  : {LLM_BACKEND}")
print(f"  • LLM_MODEL    : {LLM_MODEL or ('Ternary-Bonsai-2.5-27B (bonsai-2-27b)' if LLM_BACKEND == 'ninfer' else '(按 GPU 显存自动推荐)')}")
print(f"  • EMBED_MODEL  : {EMBED_MODEL} (device={EMBED_DEVICE})")
print(f"  • TUNNEL_MODE  : {TUNNEL_MODE}" + (f" (host={os.environ['CIVIC_COLAB_HOSTNAME']})" if TUNNEL_MODE == "named" else ""))
print(f"  • PORTS        : LLM={LLM_PORT}, EMBED={EMBED_PORT}")
print(f"  • BEARER_TOKEN : 已设置 ({masked}，启动后将在下方打印完整接入配置)")

## Step 4：拉起 `serve_colab.py` 并获取系统接入配置

执行本单元后将依次完成：
1. 在 `8133` 端口拉起 `BAAI/bge-m3` OpenAI 兼容向量服务（同时暴露 `/v1/*` 与 `/embed/v1/*`）；
2. 在 `8132` 端口拉起 System-2 LLM 服务（`ninfer` 模式首次会下载 ~1.4 GB 引擎与 ~7.9 GB Bonsai 2.5 权重，约需 1~3 分钟）；
3. 打开隧道：若为 `named` 模式则自动绑定固定单域名 `https://civic-colab.firetable.tech/v1` + `/embed/v1`（已配置 `.env.vps` 时免复制粘贴）；否则输出可直接填入 **「民声智理后台 → 多站点与算力管理」** 的接入配置。

> 保持本单元处于运行状态即可持续为「民声智理」提供远程 GPU 算力。

In [ ]:
#@title 启动双引擎算力节点并打印接入面板配置
import os, pathlib, signal, subprocess, sys

candidates = [
    pathlib.Path("serve_colab.py"),
    pathlib.Path("scripts/colab/serve_colab.py"),
    pathlib.Path("/content/serve_colab.py"),
]
script_path = next((p for p in candidates if p.is_file()), None)
if not script_path:
    raise FileNotFoundError("未找到 serve_colab.py，请先完成 Step 2 上传或克隆仓库。")

if "COLAB_API_KEY" not in os.environ:
    raise RuntimeError("请先执行 Step 3 生成或配置 COLAB_API_KEY。")

cmd = [
    sys.executable,
    str(script_path),
    "--llm-backend", LLM_BACKEND,
    "--embed-model", EMBED_MODEL,
    "--llm-port", str(LLM_PORT),
    "--embed-port", str(EMBED_PORT),
    "--tunnel", TUNNEL_MODE,
    "--embed-device", EMBED_DEVICE,
]
if TUNNEL_MODE == "named":
    cmd.extend(["--public-host", os.environ.get("CIVIC_COLAB_HOSTNAME", "civic-colab.firetable.tech")])
if LLM_MODEL.strip():
    cmd.extend(["--llm-model", LLM_MODEL.strip()])

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
except KeyboardInterrupt:
    print("\n收到中断信号，正在通知 serve_colab.py 清理子进程...")
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=10)

## Step 5：用完即释与安全注意事项

1. **用完即释**：研判演示或批量任务结束后，点击 Step 4 左侧停止按钮中断服务，随后在 Colab 菜单选择 **「代码执行程序 → 断开连接并删除运行时」**，避免持续消耗计算单元（CU）。
2. **后台清理**：回到「民声智理后台 → 多站点与算力管理」面板，停用或删除已关闭的临时 Colab 节点，防止系统持续探活。
3. **密钥安全**：不要保存包含 Step 4 输出（含 Bearer Token 与临时隧道 URL）的 `.ipynb` 副本到公开代码库。